In [2]:
# Load environment variables from project root
from pathlib import Path
from dotenv import load_dotenv

for _dir in [Path.cwd(), *Path.cwd().parents]:
    _env = _dir / '.env'
    if _env.is_file():
        load_dotenv(_env)
        break

In [3]:
# Create Databricks Spark session
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()
spark.sql('CREATE SCHEMA IF NOT EXISTS ddc_databricks.silver')

DataFrame[]

In [4]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read bronze source tables
bronze_repos = spark.table('ddc_databricks.bronze.github_repos')
bronze_contributors = spark.table('ddc_databricks.bronze.github_contributors')
bronze_events = spark.table('ddc_databricks.bronze.github_events')
bronze_readmes = spark.table('ddc_databricks.bronze.github_readmes')

In [6]:
# 1) Repos -> silver.github_repos
repos_clean = (
    bronze_repos
    .select(
        F.col('_pypi_package').alias('package_name'),
        F.to_timestamp('_ingested_at').alias('ingested_at'),
        F.col('_endpoint').alias('source_endpoint'),
        F.col('data.id').cast('long').alias('repo_id'),
        F.lower(F.col('data.full_name')).alias('repo_full_name'),
        F.col('data.name').alias('repo_name'),
        F.col('data.owner.login').alias('repo_owner'),
        F.col('data.description').alias('description'),
        F.lower(F.col('data.language')).alias('primary_language'),
        F.col('data.license.spdx_id').alias('license_spdx'),
        F.col('data.stargazers_count').cast('long').alias('stars'),
        F.col('data.forks_count').cast('long').alias('forks'),
        F.col('data.open_issues_count').cast('long').alias('open_issues'),
        F.col('data.subscribers_count').cast('long').alias('watchers'),
        F.col('data.default_branch').alias('default_branch'),
        F.to_timestamp(F.col('data.created_at')).alias('repo_created_at'),
        F.to_timestamp(F.col('data.updated_at')).alias('repo_updated_at'),
        F.to_timestamp(F.col('data.pushed_at')).alias('last_push_at'),
        F.coalesce(F.col('data.archived'), F.lit(False)).cast('boolean').alias('is_archived'),
        F.coalesce(F.col('data.disabled'), F.lit(False)).cast('boolean').alias('is_disabled'),
        F.coalesce(F.col('data.fork'), F.lit(False)).cast('boolean').alias('is_fork'),
        F.coalesce(F.col('data.has_wiki'), F.lit(False)).cast('boolean').alias('has_wiki'),
        F.col('data.topics').alias('topics')
    )
    .withColumn('repo_age_days', F.datediff(F.current_date(), F.to_date('repo_created_at')))
)

repos_dedup_w = Window.partitionBy('package_name').orderBy(F.col('ingested_at').desc())
repos_latest = (
    repos_clean
    .withColumn('rn', F.row_number().over(repos_dedup_w))
    .filter(F.col('rn') == 1)
    .drop('rn')
)

(
    repos_latest
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.silver.github_repos')
)

print('silver.github_repos created')

silver.github_repos created


In [11]:
# 2) Contributors -> silver.github_contributors + silver.github_contributor_stats
contributors_flat = (
    bronze_contributors
    .select(
        F.col('_pypi_package').alias('package_name'),
        F.to_timestamp('_ingested_at').alias('ingested_at'),
        F.explode_outer('contributors').alias('c')
    )
    .select(
        'package_name',
        'ingested_at',
        F.coalesce(F.col('c.id').cast('string'), F.col('c.login')).alias('contributor_key'),
        F.col('c.id').cast('long').alias('contributor_id'),
        F.col('c.login').alias('contributor_login'),
        F.col('c.type').alias('contributor_type'),
        F.col('c.site_admin').cast('boolean').alias('is_site_admin'),
        F.col('c.contributions').cast('long').alias('contributions')
    )
    .filter(F.col('contributor_key').isNotNull())
)

contrib_dedup_w = Window.partitionBy('package_name', 'contributor_key').orderBy(
    F.col('ingested_at').desc(),
    F.col('contributions').desc()
)

contributors_latest = (
    contributors_flat
    .withColumn('rn', F.row_number().over(contrib_dedup_w))
    .filter(F.col('rn') == 1)
    .drop('rn')
)

(
    contributors_latest
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.silver.github_contributors')
)

contributor_stats = (
    contributors_latest
    .groupBy('package_name')
    .agg(
        F.countDistinct('contributor_key').alias('contributors_total'),
        F.sum(F.coalesce(F.col('contributions'), F.lit(0))).alias('contributions_sum'),
        F.sum(F.when(F.col('contributions') >= 5, 1).otherwise(0)).alias('core_contributors_5plus'),
        F.max('contributions').alias('top_contributor_contributions')
    )
    .withColumn(
        'top_contributor_share',
        F.when(F.col('contributions_sum') > 0, F.col('top_contributor_contributions') / F.col('contributions_sum')).otherwise(F.lit(None))
    )
)

(
    contributor_stats
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.silver.github_contributor_stats')
)

print('silver.github_contributors and silver.github_contributor_stats created')

silver.github_contributors and silver.github_contributor_stats created


In [8]:
# 3) Events -> silver.github_events + silver.github_event_features
repos_lookup = repos_latest.select('package_name', F.lower('repo_full_name').alias('repo_full_name')).dropDuplicates()

events_flat = (
    bronze_events
    .select(F.to_timestamp('_ingested_at').alias('ingested_at'), F.explode_outer('events').alias('event'))
    .withColumn('payload_json', F.to_json(F.col('event.payload')))
    .select(
        'ingested_at',
        F.col('event.id').cast('string').alias('event_id'),
        F.col('event.type').alias('event_type'),
        F.to_timestamp(F.col('event.created_at')).alias('event_created_at'),
        F.lower(F.col('event.repo.name')).alias('repo_full_name'),
        F.col('event.actor.login').alias('actor_login'),
        F.col('event.public').cast('boolean').alias('is_public'),
        F.get_json_object(F.col('payload_json'), '$.action').alias('payload_action'),
        F.get_json_object(F.col('payload_json'), '$.ref').alias('payload_ref'),
        F.get_json_object(F.col('payload_json'), '$.ref_type').alias('payload_ref_type'),
        F.get_json_object(F.col('payload_json'), '$.size').cast('long').alias('payload_size'),
        F.get_json_object(F.col('payload_json'), '$.distinct_size').cast('long').alias('payload_distinct_size')
    )
    .join(repos_lookup, on='repo_full_name', how='inner')
    .withColumn(
        'event_surrogate_key',
        F.sha2(F.concat_ws('||', F.coalesce('event_id', F.lit('')), 'repo_full_name', F.coalesce('event_type', F.lit('')), F.coalesce(F.col('event_created_at').cast('string'), F.lit(''))), 256)
    )
)

events_dedup_w = Window.partitionBy('event_surrogate_key').orderBy(F.col('ingested_at').desc())
events_latest = (
    events_flat
    .withColumn('rn', F.row_number().over(events_dedup_w))
    .filter(F.col('rn') == 1)
    .drop('rn')
)

(
    events_latest
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.silver.github_events')
)

event_features = (
    events_latest
    .groupBy('package_name')
    .agg(
        F.countDistinct('event_surrogate_key').alias('events_total'),
        F.sum(F.when(F.col('event_type') == 'WatchEvent', 1).otherwise(0)).alias('watch_events_total'),
        F.sum(F.when(F.col('event_type') == 'PushEvent', 1).otherwise(0)).alias('push_events_total'),
        F.sum(F.when(F.col('event_type') == 'PullRequestEvent', 1).otherwise(0)).alias('pr_events_total'),
        F.sum(F.when(F.col('event_type') == 'IssuesEvent', 1).otherwise(0)).alias('issues_events_total'),
        F.sum(F.when(F.col('event_type') == 'ForkEvent', 1).otherwise(0)).alias('fork_events_total'),
        F.max('event_created_at').alias('last_event_at'),
        F.sum(F.when(F.col('event_created_at') >= F.expr('current_timestamp() - INTERVAL 7 DAYS'), 1).otherwise(0)).alias('events_7d'),
        F.sum(F.when(F.col('event_created_at') >= F.expr('current_timestamp() - INTERVAL 30 DAYS'), 1).otherwise(0)).alias('events_30d')
    )
)

(
    event_features
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.silver.github_event_features')
)

print('silver.github_events and silver.github_event_features created')

silver.github_events and silver.github_event_features created


In [ ]:
# 4) Readmes -> silver.github_readmes
heading_pattern = F.lit(r'(?m)^#{1,6}\s')
code_fence_pattern = F.lit('```')
markdown_link_pattern = F.lit(r'\[[^\]]+\]\([^\)]+\)')
image_pattern = F.lit(r'!\[[^\]]*\]\([^\)]+\)')
url_pattern = F.lit(r'https?://')

readmes_clean = (
    bronze_readmes
    .select(
        F.col('_pypi_package').alias('package_name'),
        F.to_timestamp('_ingested_at').alias('ingested_at'),
        F.col('_endpoint').alias('source_endpoint'),
        F.coalesce(F.col('readme_text'), F.lit('')).cast('string').alias('readme_text')
    )
    .withColumn('readme_text', F.regexp_replace('readme_text', r'\u0000', ''))
    .withColumn('readme_text', F.regexp_replace('readme_text', r'\r\n', '\n'))
    .withColumn('readme_char_count', F.length('readme_text'))
    .withColumn('readme_word_count', F.size(F.split(F.trim('readme_text'), r'\s+')))
    .withColumn('readme_line_count', F.size(F.split('readme_text', r'\n')))
    .withColumn('heading_count', F.regexp_count(F.col('readme_text'), heading_pattern))
    .withColumn('code_block_count', F.regexp_count(F.col('readme_text'), code_fence_pattern))
    .withColumn('markdown_link_count', F.regexp_count(F.col('readme_text'), markdown_link_pattern))
    .withColumn('image_count', F.regexp_count(F.col('readme_text'), image_pattern))
    .withColumn('url_count', F.regexp_count(F.col('readme_text'), url_pattern))
)

readmes_dedup_w = Window.partitionBy('package_name').orderBy(F.col('ingested_at').desc())
readmes_latest = (
    readmes_clean
    .withColumn('rn', F.row_number().over(readmes_dedup_w))
    .filter(F.col('rn') == 1)
    .drop('rn')
)

(
    readmes_latest
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.silver.github_readmes')
)

print('silver.github_readmes created')

AnalysisException: [INVALID_ATTRIBUTE_NAME_SYNTAX] Syntax error in the attribute name: ```. Check that backticks appear in pairs, a quoted string is a complete name part and use a backtick only inside quoted name parts. SQLSTATE: 42601

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at org.apache.spark.sql.errors.DataTypeErrors$.attributeNameSyntaxError(DataTypeErrors.scala:151)
	at org.apache.spark.sql.catalyst.util.AttributeNameParser.e$1(AttributeNameParser.scala:29)
	at org.apache.spark.sql.catalyst.util.AttributeNameParser.parseAttributeName(AttributeNameParser.scala:62)
	at org.apache.spark.sql.catalyst.util.AttributeNameParser.parseAttributeName$(AttributeNameParser.scala:28)
	at org.apache.spark.sql.catalyst.analysis.UnresolvedAttribute$.parseAttributeName(unresolved.scala:360)
	at org.apache.spark.sql.catalyst.analysis.UnresolvedAttribute$.quotedString(unresolved.scala:379)
	at org.apache.spark.sql.connect.planner.SparkConnectPlanner.transformUnresolvedAttribute(Spark